# Polarization analysis (full dataset)

Louvain and Leiden community detection on the reply graph, before and after removing the accounts
BotMoE labelled as bots (paper, Sections 4.5 and 5.2), plus Graphistry visualizations of the largest
communities.

Needs an NVIDIA GPU with [RAPIDS](https://rapids.ai/) (`cudf`, `cugraph`). For a CPU version that runs on
the sample data, see `polarization_sample.ipynb`.

In [ ]:
import os
import pandas as pd
%matplotlib inline

# Paths to the BotMoE predictions and to the node.json / edge.csv they were made from
PREDICTIONS = "../BotMoE/artifacts/predictions/test_predictions.csv"
DATA_DIR = "../preprocessing/data"

In [ ]:
# Read IDs as strings: as numbers, 19-digit IDs get rounded.
predictions = pd.read_csv(PREDICTIONS, dtype={"user_id": str})
predictions["prediction"].value_counts()

In [ ]:
edges = pd.read_csv(f"{DATA_DIR}/edge.csv", dtype={'source_id': str, 'target_id': str})
# The raw dataset stores reply targets as floats, so all IDs are matched as floats from here on.
edges['source_id'] = edges['source_id'].astype(dtype=float)
edges['target_id'] = edges['target_id'].astype(float)

In [ ]:
nodes = pd.read_json(f"{DATA_DIR}/node.json", dtype={"id": str})
nodes.shape

In [ ]:
edges.shape

In [ ]:
labelled_nodes = nodes.merge(predictions, left_on="id", right_on="user_id")
labelled_nodes = labelled_nodes.rename(columns={'prediction': 'label'})
# Join on exact IDs above, then switch to float keys to match edge.csv.
labelled_nodes['id'] = labelled_nodes['id'].astype(float)

In [ ]:
import cugraph

In [ ]:
nodes_df = labelled_nodes[["id", "label"]]

In [ ]:
pd.set_option('display.max_colwidth', None)

### Community detection (cuGraph)

In [ ]:
import cudf
import cugraph

edges_df = edges.drop(columns=['relation']).rename(columns={'source_id': 'src', 'target_id': 'dst'})
# its important to leave IDs as float otherwise matching will be inaccurate
# labelled_nodes['id'] = labelled_nodes['id'].astype('str')
# edges_df['src'] = edges_df['src'].astype('str')
# edges_df['dst'] = edges_df['dst'].astype('str')
print("formatted edges df, converting to cudf")
# Convert to cuDF
edges_gdf = cudf.DataFrame.from_pandas(edges_df)

In [ ]:
valid_edges_gdf = edges_gdf[edges_gdf["dst"].isin(labelled_nodes["id"])]
print("num of edges before cleaning", edges_gdf.shape)
print("num of edges after cleaning", valid_edges_gdf.shape)


In [ ]:
# Create cuGraph Graph
G_cu = cugraph.Graph()
G_cu.from_cudf_edgelist(valid_edges_gdf, source='src', destination='dst')
# G_cu = G
# Run Louvain community detection
parts, modularity_score = cugraph.louvain(G_cu)


print(f"Modularity (before removing bots): {modularity_score}")
print("Number of detected communities (after removing bots): ", parts["partition"].nunique())
# Find bot IDs
bot_ids = set(nodes_df.query("label == 1")['id'])

# Drop edges where either src or dst is a bot
edges_no_bots = valid_edges_gdf[
    ~valid_edges_gdf['src'].isin(bot_ids) & ~valid_edges_gdf['dst'].isin(bot_ids)
]

# To cuDF
edges_no_bots_gdf = cudf.DataFrame.from_pandas(edges_no_bots.to_pandas())

# New cuGraph
G_cu_no_bots = cugraph.Graph()
G_cu_no_bots.from_cudf_edgelist(edges_no_bots_gdf, source='src', destination='dst')

# Louvain again
parts_no_bots, modularity_no_bots = cugraph.louvain(G_cu_no_bots)

print(f"Modularity (after removing bots): {modularity_no_bots}")
print("Number of detected communities (after removing bots): ", parts_no_bots["partition"].nunique())


In [ ]:
nodes_df_gdf = cudf.DataFrame.from_pandas(nodes_df)
parts_with_labels = parts.merge(nodes_df_gdf.rename(columns={'id': 'vertex'}), on='vertex', how='left')

parts_with_labels['label'] = parts_with_labels['label'].fillna(0)

community_stats = parts_with_labels.groupby('partition').agg({
    'vertex': 'count',       # community size
    'label': 'sum'           # number of bots (sum of 1s)
}).rename(columns={'vertex': 'community_size', 'label': 'num_bots'})
community_stats['bot_percentage'] = (community_stats['num_bots'] / community_stats['community_size']) * 100
community_stats = community_stats.sort_values('community_size', ascending=False)

community_stats_df = community_stats.to_pandas()

print(community_stats_df.head(50))


In [ ]:
import cudf
import cugraph

def plot_community_distribution(parts):
    community_sizes = parts.groupby('partition').size().reset_index(name='size')

    community_sizes = community_sizes.sort_values(by='size', ascending=False)

    print(community_sizes.tail(10))

    community_sizes.to_pandas().to_csv('community_sizes.csv', index=False)

    import matplotlib.pyplot as plt

    sizes = community_sizes['size'].to_pandas()

    plt.figure(figsize=(10,6))
    plt.hist(sizes, bins=100, log=True)
    plt.xlabel('Community size')
    plt.ylabel('Frequency (log scale)')
    plt.title('Distribution of Community Sizes')
    plt.show()


In [ ]:
plot_community_distribution(parts)

In [ ]:
parts_l, modularity_score_l = cugraph.leiden(G_cu, random_state=1)
print(f"Modularity (Leiden): {modularity_score_l}")
print("Number of communities:", parts_l["partition"].nunique())


In [ ]:
parts_l_humans, modularity_score_l_humans = cugraph.leiden(G_cu_no_bots, random_state=1)
print(f"Modularity (Leiden): {modularity_score_l_humans}")
print("Number of communities:", parts_l_humans["partition"].nunique())

### Graphistry

In [ ]:
import graphistry

In [ ]:
# Set GRAPHISTRY_KEY_ID and GRAPHISTRY_SECRET to your Graphistry personal key.
graphistry.register(api=3, personal_key_id=os.environ["GRAPHISTRY_KEY_ID"],
                    personal_key_secret=os.environ["GRAPHISTRY_SECRET"])

#### User level graph

In [ ]:
import cugraph

G = cugraph.Graph()
G.from_cudf_edgelist(valid_edges_gdf, source='src', destination='dst', renumber=True)

print("Running Louvain...")
parts, modularity = cugraph.louvain(G)

community_sizes = parts['partition'].value_counts()
print(f"Total communities found: {len(community_sizes)}")
top_10_clusters = community_sizes.head(3).index.to_arrow().to_pylist()
print(f"Top 10 cluster IDs: {top_10_clusters}")

users_in_top_10 = parts[parts['partition'].isin(top_10_clusters)]

filtered_nodes = labelled_nodes[
    (labelled_nodes['id'].isin(users_in_top_10['vertex'].to_pandas())) & 
    (labelled_nodes['label'] == 1)
][["id", "label"]]

edges_filtered = valid_edges_gdf[
    (valid_edges_gdf['src'].isin(users_in_top_10['vertex'].to_pandas())) & 
    (valid_edges_gdf['dst'].isin(users_in_top_10['vertex'].to_pandas()))
]

valid_ids = filtered_nodes['id'].to_list()

edges_filtered = valid_edges_gdf[
    (valid_edges_gdf['src'].isin(valid_ids)) & 
    (valid_edges_gdf['dst'].isin(valid_ids))
]
filtered_nodes['id'] = filtered_nodes['id'].astype(str)
edges_filtered['src'] = edges_filtered['src'].astype(str)
edges_filtered['dst'] = edges_filtered['dst'].astype(str)

print(f"Filtered nodes count: {len(filtered_nodes)}")
print(f"Filtered edges count: {len(edges_filtered)}")

# Step 7: Visualize user-level graph with Graphistry
print("Visualizing user-level graph for top 50 communities...")
plot = (
    graphistry
    .bind(source='src', destination='dst', node='id')  # 'id' is the node ID column in filtered_nodes
    .nodes(filtered_nodes)
    .edges(edges_filtered)
    .plot()
)


In [ ]:
labelled_nodes.shape

In [ ]:
community_sizes

In [ ]:
community_sizes_percent = (community_sizes / labelled_nodes.shape[0]) * 100
community_sizes_percent[:50]

In [ ]:
nodes_with_parts = filtered_nodes.merge(
    parts.to_pandas(),
    left_on='id',
    right_on='vertex',
    how='left'
)
graphistry.bind(source='src', destination='dst', node='id').nodes(nodes_with_parts).edges(valid_edges_gdf).plot()


In [ ]:
# plot

#### Community level graph

In [ ]:
labelled_nodes[labelled_nodes["id"] == 1521743190247804928].username

In [ ]:
import cugraph

# Step 1: Build graph
G = cugraph.Graph()
G.from_cudf_edgelist(valid_edges_gdf, source='src', destination='dst', renumber=True)

# Step 2: Louvain
print("Running Louvain...")
parts, modularity = cugraph.louvain(G)

# Step 3: Top 10 clusters
community_sizes = parts['partition'].value_counts()
print(f"Total communities: {len(community_sizes)}")
top_10_clusters = community_sizes.head(10).index.to_arrow().to_pylist()
print(f"Top 10 cluster IDs: {top_10_clusters}")

# Step 4: Add cluster info to edges
edges_with_clusters = valid_edges_gdf.merge(
    parts, left_on='src', right_on='vertex', how='left'
).rename(columns={'partition': 'src_cluster'})

edges_with_clusters = edges_with_clusters.merge(
    parts, left_on='dst', right_on='vertex', how='left'
).rename(columns={'partition': 'dst_cluster'})

# Step 5: Filter to edges where both ends in top 10 clusters
edges_with_clusters_top = edges_with_clusters[
    edges_with_clusters['src_cluster'].isin(top_10_clusters) &
    edges_with_clusters['dst_cluster'].isin(top_10_clusters)
]

# Step 6: Aggregate to cluster edges
cluster_edges = (
    edges_with_clusters_top
    .groupby(['src_cluster', 'dst_cluster'])
    .size()
    .reset_index(name='edge_weight')
)

# Step 7: Build cluster nodes dataframe
community_nodes = parts[parts['partition'].isin(top_10_clusters)][['partition']].drop_duplicates()
community_nodes = community_nodes.rename(columns={'partition': 'community'})

# Step 8: Convert IDs to string
cluster_edges['src_cluster'] = cluster_edges['src_cluster'].astype(str)
cluster_edges['dst_cluster'] = cluster_edges['dst_cluster'].astype(str)
community_nodes['community'] = community_nodes['community'].astype(str)

# Step 9: Visualize
print("Visualizing community-level graph...")
plot = (
    graphistry
    .bind(source='src_cluster', destination='dst_cluster', node='community')
    .nodes(community_nodes)
    .edges(cluster_edges)
    .plot()
)


In [ ]:
nodes_with_parts = labelled_nodes.merge(
    parts.to_pandas(),
    left_on='id',
    right_on='vertex',
    how='left'
)
